In [90]:
import itertools
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import seaborn as sns
from tqdm import tqdm

out_dir = Path("out")
out_dir.mkdir(exist_ok=True, parents=True)

pl.Config.set_tbl_rows(100)

train_df = pl.read_csv("../../data/raw/train.csv")
test_df = pl.read_csv("../../data/raw/test.csv")
submit_df = pl.read_csv("../../data/raw/sample_submission.csv")

train_df = train_df.drop("id").cast({"date": pl.Date})
test_df = test_df.drop("id").cast({"date": pl.Date})

train_df.head()

date,country,store,product,num_sold
date,str,str,str,f64
2010-01-01,"""Canada""","""Discount Stickers""","""Holographic Goose""",null
2010-01-01,"""Canada""","""Discount Stickers""","""Kaggle""",973.0
2010-01-01,"""Canada""","""Discount Stickers""","""Kaggle Tiers""",906.0
2010-01-01,"""Canada""","""Discount Stickers""","""Kerneler""",423.0
2010-01-01,"""Canada""","""Discount Stickers""","""Kerneler Dark Mode""",491.0


In [91]:
gdp_df = pl.read_csv("../../data/external/gdp.csv")
gdp_df

year,Canada,Finland,Italy,Kenya,Norway,Singapore
i64,f64,f64,f64,f64,f64,f64
2010,47560.666601,46506.291902,36184.71187,1091.518731,88163.208593,47236.683085
2011,52223.85884,51060.324259,38851.388134,1096.144998,101221.813477,53891.457026
2012,52670.344734,47551.674084,35235.798906,1285.005362,102175.919298,55547.555308
2013,52638.118724,49691.01452,35747.707953,1370.878852,103553.840134,56967.425794
2014,50960.843117,50073.776082,35750.71975,1482.815663,97666.695184,57564.802311
2015,43594.194105,42560.345677,30387.129319,1489.119598,74809.965805,55645.606861
2016,42314.061582,43451.256244,31126.324695,1554.126103,70867.360997,56899.918181
2017,45129.628117,46085.017474,32554.146685,1667.484472,76131.838403,61162.097393
2018,46539.176157,49654.249704,34746.344139,1836.452755,82792.842711,66840.637339


In [92]:
countries = train_df["country"].unique().to_list()
stores = train_df["store"].unique().to_list()
products = train_df["product"].unique().to_list()

train_df = train_df.with_columns(
    pl.col("date").dt.year().alias("year"),
    pl.col("date").dt.month().alias("month"),
    pl.col("date").dt.week().alias("week"),
    pl.col("date").dt.weekday().alias("weekday"),
    pl.col("date").dt.day().alias("day"),
).join(
    gdp_df.unpivot(countries, index="year", variable_name="country", value_name="gdp"),
    on=["year", "country"],
)

train_df.head()

date,country,store,product,num_sold,year,month,week,weekday,day,gdp
date,str,str,str,f64,i32,i8,i8,i8,i8,f64
2010-01-01,"""Canada""","""Discount Stickers""","""Holographic Goose""",null,2010,1,53,5,1,47560.666601
2010-01-01,"""Canada""","""Discount Stickers""","""Kaggle""",973.0,2010,1,53,5,1,47560.666601
2010-01-01,"""Canada""","""Discount Stickers""","""Kaggle Tiers""",906.0,2010,1,53,5,1,47560.666601
2010-01-01,"""Canada""","""Discount Stickers""","""Kerneler""",423.0,2010,1,53,5,1,47560.666601
2010-01-01,"""Canada""","""Discount Stickers""","""Kerneler Dark Mode""",491.0,2010,1,53,5,1,47560.666601


In [93]:
out_dir = Path("./out/gdp_vs_num_sold")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir(exist_ok=True, parents=True)
for category in ["country", "store", "product"]:
    tmp = (
        train_df.drop_nulls()
        .group_by("year", "country", "store", "product")
        .agg(pl.col("num_sold").mean(), pl.col("gdp").mean())
    )
    fig, ax = plt.subplots(figsize=(10, 10))
    sns.scatterplot(
        tmp, x="gdp", y="num_sold", hue=category, edgecolor=None, s=5, ax=ax
    )
    ax.set_title(f"{category} vs num_sold vs GDP")
    ax.set_xlabel("GDP")
    ax.set_ylabel("num_sold")
    plt.savefig(out_dir / f"{category}.png")
    fig.clear()
    plt.close(fig)


In [109]:
from sklearn.linear_model import LinearRegression

df = (
    train_df.drop_nulls()
    .group_by("year", "country", "store", "product", maintain_order=True)
    .agg(pl.col("num_sold").mean(), pl.col("gdp").mean())
)
df.group_by("year", "store", "product", maintain_order=True).agg(
    pl.col("num_sold").mean(), pl.col("gdp").mean()
)

coef_dict = {}
stores = ["Discount Stickers", "Stickers for Less", "Premium Sticker Mart"]
products = [
    "Holographic Goose",
    "Kerneler",
    "Kerneler Dark Mode",
    "Kaggle Tiers",
    "Kaggle",
]
for store, product in itertools.product(stores, products):
    tmp = df.filter(pl.col("store") == store).filter(pl.col("product") == product)
    X = tmp.select("gdp").to_numpy()
    y = tmp["num_sold"].to_numpy()
    reg = LinearRegression(fit_intercept=False).fit(X, y)
    coef = reg.coef_[0]
    coef_dict.setdefault("store", []).append(store)
    coef_dict.setdefault("product", []).append(product)
    coef_dict.setdefault("coef", []).append(coef)

coef_df = pl.DataFrame(coef_dict)
coef_df.write_csv("../../data/external/gdp_vs_num_sold_linear_coef.csv")
coef_df = coef_df.pivot("store", index="product", values="coef")

xlabels = coef_df.drop("product").columns
ylabels = coef_df["product"].to_list()
m = coef_df.drop("product").to_numpy()

out_dir = Path("./out")
fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(
    m, xticklabels=xlabels, yticklabels=ylabels, annot=True, cmap="viridis", ax=ax
)
ax.set_title("GDP coef")
ax.set_xlabel("store")
ax.set_ylabel("product")
fig.savefig(out_dir / "gdp_vs_num_sold_coef.png")
fig.clear()
plt.close(fig)


In [ ]:
coef_df = pl.DataFrame(coef_dict)

out_dir = Path("./out/gdp_vs_num_sold_reg")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir(exist_ok=True, parents=True)
for store, product in itertools.product(stores, products):
    tmp = (
        train_df.drop_nulls()
        .group_by("year", "country", "store", "product")
        .agg(pl.col("num_sold").mean(), pl.col("gdp").mean())
        .with_columns(
            (pl.col("store").eq(store) & pl.col("product").eq(product)).alias(
                f"{store}_{product}"
            )
        )
    )
    co = float(
        coef_df.filter(pl.col("store") == store)
        .filter(pl.col("product") == product)["coef"]
        .item()
    )

    fig, ax = plt.subplots(figsize=(10, 10))
    sns.scatterplot(
        tmp,
        x="gdp",
        y="num_sold",
        hue=f"{store}_{product}",
        edgecolor=None,
        s=5,
        ax=ax,
    )
    ax.plot(
        [0, tmp["gdp"].max()],  # type: ignore
        [0, co * tmp["gdp"].max()],  # type: ignore
    )
    ax.set_title(f"{store} {product} vs num_sold vs GDP")
    ax.set_xlabel("GDP")
    ax.set_ylabel("num_sold")
    fig.savefig(out_dir / f"{store}_{product}.png")
    fig.clear()
    plt.close(fig)


In [95]:
from sklearn.metrics import mean_squared_error, r2_score

coef_df = pl.DataFrame(coef_dict)
coef_df = coef_df.to_dummies(["store", "product"])

X = coef_df.drop("coef").to_numpy()
y = coef_df["coef"].to_numpy()
reg = LinearRegression(fit_intercept=False).fit(X, y)
y_pred = reg.predict(X)
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(
    f"Mean squared error: {mse}\n"
    f"R2 score: {r2}\n"
    f"Intercept: {reg.intercept_}\n"
    f"Coefficients: {reg.coef_}"
)

# 係数を棒グラフで表示
fig, ax = plt.subplots(figsize=(10, 10))
sns.barplot(x=reg.coef_, y=coef_df.drop("coef").columns, ax=ax)
ax.set_title("GDP coef")
ax.set_xlabel("store_product")
ax.set_ylabel("coef")
fig.savefig(out_dir / "gdp_vs_num_sold_coef_bar.png")
fig.clear()


Mean squared error: 6.699453964361088e-06
R2 score: 0.9301989522385535
Intercept: 0.0
Coefficients: [ 0.00280123  0.01485134  0.01168131 -0.00604038  0.01677332  0.0121545
  0.00220033  0.00424611]


<Figure size 1000x1000 with 0 Axes>

In [96]:
coef_df = pl.DataFrame(coef_dict)
coef_df = coef_df.with_columns(pl.Series("coef_pred", y_pred))
coef_df.with_columns((pl.col("coef") - pl.col("coef_pred")).abs().alias("abs_diff"))

store,product,coef,coef_pred,abs_diff
str,str,f64,f64,f64
"""Discount Stickers""","""Holographic Goose""",0.002052,-0.003239,0.005291
"""Discount Stickers""","""Kerneler""",0.006636,0.005002,0.001634
"""Discount Stickers""","""Kerneler Dark Mode""",0.007769,0.007047,0.000721
"""Discount Stickers""","""Kaggle Tiers""",0.012158,0.014956,0.002798
"""Discount Stickers""","""Kaggle""",0.014726,0.019575,0.004849
"""Stickers for Less""","""Holographic Goose""",0.004226,0.005641,0.001414
"""Stickers for Less""","""Kerneler""",0.01344,0.013882,0.000441
"""Stickers for Less""","""Kerneler Dark Mode""",0.015727,0.015927,0.000201
"""Stickers for Less""","""Kaggle Tiers""",0.024583,0.023836,0.000747


In [118]:
coef_df = pl.DataFrame(coef_dict)
out_dir = Path("./out/num_sold_over_date_adj")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir(exist_ok=True, parents=True)
for category in ["country", "store", "product"]:
    tmp = (
        train_df.join(coef_df, on=["store", "product"])
        .with_columns(
            (pl.col("gdp") * pl.col("coef")).alias("gdp_linear_pred"),
        )
        .with_columns(
            (pl.col("num_sold") - pl.col("gdp_linear_pred")).alias("num_sold_adj")
        )
        .drop_nulls()
        .group_by("date", category)
        .agg(pl.col("num_sold_adj").mean())
    )
    fig, ax = plt.subplots(figsize=(20, 8))
    sns.lineplot(data=tmp, x="date", y="num_sold_adj", hue=category, ax=ax)
    ax.set_title(f"num_sold_adj over date by {category}")
    ax.set_xlabel("date")
    ax.set_ylabel("num_sold_adj")
    fig.savefig(out_dir / f"{category}.png")
    fig.clear()
    plt.close(fig)

In [120]:
coef_df = pl.DataFrame(coef_dict)
(
    train_df.join(coef_df, on=["store", "product"])
    .with_columns(
        (pl.col("gdp") * pl.col("coef")).alias("gdp_linear_pred"),
    )
    .with_columns(
        (pl.col("num_sold") - pl.col("gdp_linear_pred")).alias("num_sold_adj")
    )
    .write_parquet("../../data/processed/train_num_sold_adj.parquet")
)